# Gold Layer - Business KPIs and Driver Performance Analytics

## Objective

The Gold Layer converts the cleaned Silver dataset into business-ready analytical tables.

## Gold Layer Responsibilities

- Generate driver performance metrics
- Calculate cancellation rates
- Analyze revenue
- Analyze delays
- Identify high-demand pickup locations
- Rank drivers using Spark window functions
- Store final analytical outputs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_path = "/Volumes/workspace/default/rideshare_data/silver/ride_data"

silver_df = spark.read.parquet(silver_path)

print("Silver records:", silver_df.count())
display(silver_df.limit(10))

Silver records: 150


trip_id,driver_id,driver_name,city,rating,pickup_location,drop_location,distance_km,fare_amount,trip_status,log_id,start_time,end_time,delay_minutes,cancellation_flag,trip_duration_minutes,completion_flag,is_delayed,revenue_per_km,delay_category,trip_date,trip_hour
12,62,Anjali_62,Mumbai,4.4,Railway Station,Mall,8.32,0.0,Cancelled,12,2025-01-06T14:05:00.000Z,null,0.0,1,null,0,0,0.0,No Delay,2025-01-06,14
18,57,Priya_57,Pune,4.6,Airport,IT Park,13.23,0.0,Cancelled,18,2025-01-07T15:21:00.000Z,null,0.0,1,null,0,0,0.0,No Delay,2025-01-07,15
38,143,Anjali_143,Mumbai,4.0,Railway Station,IT Park,2.23,32.64,Completed,38,2025-01-04T04:48:00.000Z,2025-01-04T05:39:00.000Z,19.0,0,51.0,1,1,14.64,High Delay,2025-01-04,4
67,81,Amit_81,Mumbai,4.1,City Center,Airport,7.81,0.0,Cancelled,67,2025-01-04T20:30:00.000Z,null,0.0,1,null,0,0,0.0,No Delay,2025-01-04,20
70,125,Neha_125,Hyderabad,3.5,Railway Station,Mall,4.79,70.95,Completed,70,2025-01-06T08:18:00.000Z,2025-01-06T08:52:00.000Z,9.0,0,34.0,1,1,14.81,Medium Delay,2025-01-06,8
93,61,Rahul_61,Bangalore,3.5,Mall,Airport,10.77,0.0,Cancelled,93,2025-01-05T03:33:00.000Z,null,0.0,1,null,0,0,0.0,No Delay,2025-01-05,3
16,150,Sneha_150,Pune,4.8,IT Park,Airport,23.76,0.0,Cancelled,16,2025-01-07T02:27:00.000Z,null,0.0,1,null,0,0,0.0,No Delay,2025-01-07,2
64,150,Sneha_150,Pune,4.8,IT Park,Mall,12.03,164.81,Completed,64,2025-01-04T00:51:00.000Z,2025-01-04T01:24:00.000Z,19.0,0,33.0,1,1,13.7,High Delay,2025-01-04,0
74,38,Sneha_38,Pune,5.0,Airport,Mall,10.82,156.38,Completed,74,2025-01-02T21:47:00.000Z,2025-01-02T22:47:00.000Z,8.0,0,60.0,1,1,14.45,Medium Delay,2025-01-02,21
94,102,Vikas_102,Delhi,4.8,City Center,IT Park,8.31,99.4,Completed,94,2025-01-02T23:52:00.000Z,2025-01-03T00:05:00.000Z,1.0,0,13.0,1,1,11.96,Low Delay,2025-01-02,23


In [0]:
driver_performance_df = (
    silver_df
    .groupBy(
        "driver_id",
        "driver_name",
        "city",
        "rating"
    )
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.sum("completion_flag").alias("completed_trips"),
        F.sum("cancellation_flag").alias("cancelled_trips"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration")
    )
    .withColumn(
        "completion_rate",
        F.round(
            (F.col("completed_trips") / F.col("total_trips")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            (F.col("cancelled_trips") / F.col("total_trips")) * 100,
            2
        )
    )
)

display(driver_performance_df)

driver_id,driver_name,city,rating,total_trips,completed_trips,cancelled_trips,avg_delay_minutes,total_revenue,avg_fare,avg_trip_duration,completion_rate,cancellation_rate
74,Priya_74,Pune,4.2,2,0,2,0.0,0.0,0.0,null,0.0,100.0
1,Rahul_1,Delhi,4.6,1,1,0,2.0,124.76,124.76,28.0,100.0,0.0
63,Amit_63,Delhi,4.6,1,1,0,13.0,334.21,334.21,41.0,100.0,0.0
117,Neha_117,Pune,3.6,1,0,1,0.0,0.0,0.0,null,0.0,100.0
17,Rahul_17,Delhi,4.5,1,0,1,0.0,0.0,0.0,null,0.0,100.0
141,Arjun_141,Mumbai,3.9,1,0,1,0.0,0.0,0.0,null,0.0,100.0
142,Sneha_142,Delhi,4.3,1,1,0,16.0,25.08,25.08,37.0,100.0,0.0
23,Anjali_23,Mumbai,3.7,1,0,1,0.0,0.0,0.0,null,0.0,100.0
49,Arjun_49,Bangalore,4.2,1,0,1,0.0,0.0,0.0,null,0.0,100.0
77,Anjali_77,Delhi,3.6,2,1,1,6.5,189.57,94.79,6.0,50.0,50.0


In [0]:
driver_rank_window = Window.orderBy(
    F.col("completion_rate").desc(),
    F.col("total_revenue").desc(),
    F.col("avg_delay_minutes").asc(),
    F.col("rating").desc()
)

driver_performance_ranked_df = (
    driver_performance_df
    .withColumn(
        "driver_rank",
        F.dense_rank().over(driver_rank_window)
    )
)

display(
    driver_performance_ranked_df.orderBy("driver_rank")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


driver_id,driver_name,city,rating,total_trips,completed_trips,cancelled_trips,avg_delay_minutes,total_revenue,avg_fare,avg_trip_duration,completion_rate,cancellation_rate,driver_rank
65,Amit_65,Bangalore,3.6,3,3,0,9.0,493.75,164.58,22.33,100.0,0.0,1
102,Vikas_102,Delhi,4.8,2,2,0,7.0,390.52,195.26,32.0,100.0,0.0,2
132,Priya_132,Hyderabad,4.0,2,2,0,19.0,387.74,193.87,20.5,100.0,0.0,3
76,Priya_76,Delhi,4.2,3,3,0,9.67,369.47,123.16,35.67,100.0,0.0,4
63,Amit_63,Delhi,4.6,1,1,0,13.0,334.21,334.21,41.0,100.0,0.0,5
87,Vikas_87,Delhi,3.6,1,1,0,18.0,236.14,236.14,33.0,100.0,0.0,6
99,Rahul_99,Delhi,4.2,1,1,0,20.0,232.49,232.49,9.0,100.0,0.0,7
38,Sneha_38,Pune,5.0,2,2,0,8.5,223.44,111.72,49.5,100.0,0.0,8
125,Neha_125,Hyderabad,3.5,2,2,0,5.0,214.39,107.19,47.0,100.0,0.0,9
54,Anjali_54,Mumbai,3.7,1,1,0,2.0,212.08,212.08,47.0,100.0,0.0,10


### Window Function Note

The overall driver ranking uses a global window without partitioning because all drivers must be compared against each other.

Spark displays a warning because global window operations move data to a single partition. This is acceptable for the current small dataset. For large-scale production data, partitioned rankings such as city-wise driver ranking should be used wherever business requirements permit.

In [0]:
city_rank_window = (
    Window
    .partitionBy("city")
    .orderBy(
        F.col("completion_rate").desc(),
        F.col("total_revenue").desc(),
        F.col("avg_delay_minutes").asc(),
        F.col("rating").desc()
    )
)

city_driver_ranked_df = (
    driver_performance_df
    .withColumn(
        "city_driver_rank",
        F.dense_rank().over(city_rank_window)
    )
)

display(
    city_driver_ranked_df.orderBy(
        "city",
        "city_driver_rank"
    )
)

driver_id,driver_name,city,rating,total_trips,completed_trips,cancelled_trips,avg_delay_minutes,total_revenue,avg_fare,avg_trip_duration,completion_rate,cancellation_rate,city_driver_rank
65,Amit_65,Bangalore,3.6,3,3,0,9.0,493.75,164.58,22.33,100.0,0.0,1
145,Rahul_145,Bangalore,4.3,1,1,0,6.0,204.85,204.85,16.0,100.0,0.0,2
24,Sneha_24,Bangalore,5.0,1,1,0,9.0,163.49,163.49,54.0,100.0,0.0,3
50,Rahul_50,Bangalore,4.8,1,1,0,2.0,122.23,122.23,50.0,100.0,0.0,4
114,Neha_114,Bangalore,3.7,4,3,1,11.25,720.44,180.11,28.33,75.0,25.0,5
103,Vikas_103,Bangalore,3.7,3,2,1,8.33,256.07,85.36,10.0,66.67,33.33,6
16,Arjun_16,Bangalore,4.4,2,1,1,1.0,211.15,105.57,14.0,50.0,50.0,7
58,Priya_58,Bangalore,4.1,2,1,1,9.0,185.6,92.8,33.0,50.0,50.0,8
21,Priya_21,Bangalore,4.6,2,1,1,1.5,51.71,25.85,59.0,50.0,50.0,9
127,Priya_127,Bangalore,3.7,3,1,2,2.33,126.14,42.05,39.0,33.33,66.67,10


In [0]:
cancellation_analysis_df = (
    silver_df
    .groupBy(
        "driver_id",
        "driver_name",
        "city"
    )
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.sum("cancellation_flag").alias("cancelled_trips")
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            (F.col("cancelled_trips") / F.col("total_trips")) * 100,
            2
        )
    )
    .orderBy(
        F.col("cancellation_rate").desc(),
        F.col("cancelled_trips").desc()
    )
)

display(cancellation_analysis_df)

driver_id,driver_name,city,total_trips,cancelled_trips,cancellation_rate
91,Rahul_91,Pune,3,3,100.0
61,Rahul_61,Bangalore,3,3,100.0
126,Vikas_126,Hyderabad,2,2,100.0
57,Priya_57,Pune,2,2,100.0
88,Rohit_88,Pune,2,2,100.0
124,Priya_124,Mumbai,2,2,100.0
81,Amit_81,Mumbai,2,2,100.0
98,Neha_98,Bangalore,2,2,100.0
139,Vikas_139,Hyderabad,2,2,100.0
108,Rahul_108,Pune,2,2,100.0


In [0]:
revenue_analysis_df = (
    silver_df
    .groupBy("city")
    .agg(
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("distance_km"), 2).alias("total_distance_km"),
        F.count("trip_id").alias("total_trips")
    )
    .withColumn(
        "revenue_per_trip",
        F.round(
            F.col("total_revenue") / F.col("total_trips"),
            2
        )
    )
    .orderBy(F.col("total_revenue").desc())
)

display(revenue_analysis_df)

city,total_revenue,avg_fare,total_distance_km,total_trips,revenue_per_trip
Delhi,2851.14,95.04,432.64,30,95.04
Bangalore,2535.43,68.53,526.65,37,68.53
Mumbai,2286.38,71.45,407.46,32,71.45
Pune,1493.44,43.92,480.54,34,43.92
Hyderabad,924.69,54.39,214.18,17,54.39


In [0]:
delay_analysis_df = (
    silver_df
    .groupBy(
        "driver_id",
        "driver_name",
        "city"
    )
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.max("delay_minutes").alias("max_delay_minutes"),
        F.sum("is_delayed").alias("delayed_trips")
    )
    .withColumn(
        "delay_rate",
        F.round(
            (F.col("delayed_trips") / F.col("total_trips")) * 100,
            2
        )
    )
    .orderBy(
        F.col("avg_delay_minutes").desc()
    )
)

display(delay_analysis_df)

driver_id,driver_name,city,total_trips,avg_delay_minutes,max_delay_minutes,delayed_trips,delay_rate
99,Rahul_99,Delhi,1,20.0,20.0,1,100.0
19,Rahul_19,Hyderabad,1,20.0,20.0,1,100.0
143,Anjali_143,Mumbai,1,19.0,19.0,1,100.0
137,Karan_137,Delhi,1,19.0,19.0,1,100.0
132,Priya_132,Hyderabad,2,19.0,20.0,2,100.0
85,Karan_85,Mumbai,1,19.0,19.0,1,100.0
111,Karan_111,Pune,1,18.0,18.0,1,100.0
87,Vikas_87,Delhi,1,18.0,18.0,1,100.0
142,Sneha_142,Delhi,1,16.0,16.0,1,100.0
45,Anjali_45,Delhi,1,16.0,16.0,1,100.0


In [0]:
high_demand_locations_df = (
    silver_df
    .groupBy("pickup_location")
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.sum("completion_flag").alias("completed_trips"),
        F.sum("cancellation_flag").alias("cancelled_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            (F.col("cancelled_trips") / F.col("total_trips")) * 100,
            2
        )
    )
    .orderBy(F.col("total_trips").desc())
)

display(high_demand_locations_df)

pickup_location,total_trips,completed_trips,cancelled_trips,total_revenue,cancellation_rate
Airport,36,13,23,2075.39,63.89
IT Park,34,17,17,3191.85,50.0
Railway Station,29,14,15,1722.94,51.72
Mall,26,13,13,2339.98,50.0
City Center,25,6,19,760.92,76.0


In [0]:
business_kpi_df = (
    silver_df
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.sum("completion_flag").alias("completed_trips"),
        F.sum("cancellation_flag").alias("cancelled_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("average_fare"),
        F.round(F.avg("delay_minutes"), 2).alias("average_delay_minutes")
    )
    .withColumn(
        "completion_rate",
        F.round(
            (F.col("completed_trips") / F.col("total_trips")) * 100,
            2
        )
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            (F.col("cancelled_trips") / F.col("total_trips")) * 100,
            2
        )
    )
)

display(business_kpi_df)

total_trips,completed_trips,cancelled_trips,total_revenue,average_fare,average_delay_minutes,completion_rate,cancellation_rate
150,63,87,10091.08,67.27,4.77,42.0,58.0


In [0]:
gold_base_path = "/Volumes/workspace/default/rideshare_data/gold"

driver_performance_ranked_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/driver_performance"
)

cancellation_analysis_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/cancellation_analysis"
)

revenue_analysis_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/revenue_analysis"
)

delay_analysis_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/delay_analysis"
)

high_demand_locations_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/high_demand_locations"
)

business_kpi_df.write.mode("overwrite").parquet(
    f"{gold_base_path}/business_kpi"
)

print("All Gold tables saved successfully.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


All Gold tables saved successfully.


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/rideshare_data/gold"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/rideshare_data/gold/business_kpi/,business_kpi/,0,1783750211768
dbfs:/Volumes/workspace/default/rideshare_data/gold/cancellation_analysis/,cancellation_analysis/,0,1783750211768
dbfs:/Volumes/workspace/default/rideshare_data/gold/delay_analysis/,delay_analysis/,0,1783750211768
dbfs:/Volumes/workspace/default/rideshare_data/gold/driver_performance/,driver_performance/,0,1783750211768
dbfs:/Volumes/workspace/default/rideshare_data/gold/high_demand_locations/,high_demand_locations/,0,1783750211768
dbfs:/Volumes/workspace/default/rideshare_data/gold/revenue_analysis/,revenue_analysis/,0,1783750211768


In [0]:
gold_tables = {
    "driver_performance": f"{gold_base_path}/driver_performance",
    "cancellation_analysis": f"{gold_base_path}/cancellation_analysis",
    "revenue_analysis": f"{gold_base_path}/revenue_analysis",
    "delay_analysis": f"{gold_base_path}/delay_analysis",
    "high_demand_locations": f"{gold_base_path}/high_demand_locations",
    "business_kpi": f"{gold_base_path}/business_kpi"
}

for table_name, table_path in gold_tables.items():
    count = spark.read.parquet(table_path).count()
    print(f"{table_name}: {count}")

driver_performance: 99
cancellation_analysis: 99
revenue_analysis: 5
delay_analysis: 99
high_demand_locations: 5
business_kpi: 1


## Gold Layer Conclusion

The Gold Layer was completed successfully.

- Driver performance metrics were generated for 99 active drivers.
- Cancellation rates were calculated for each driver.
- Revenue analysis was created across 5 cities.
- Delay analysis was generated for active drivers.
- High-demand pickup locations were identified.
- Overall business KPIs were created.
- Global and city-wise driver rankings were generated using Spark window functions.
- All Gold tables were stored successfully in Parquet format.

The analytical datasets are now ready for final validation, CSV export, GitHub submission, and Streamlit dashboard development.

In [0]:
silver_df = spark.read.parquet(
    "/Volumes/workspace/default/rideshare_data/silver/ride_data"
)

silver_df.createOrReplaceTempView("silver_ride_data")

print("Spark SQL temporary view created successfully.")

Spark SQL temporary view created successfully.


In [0]:
%sql
SELECT
    driver_id,
    driver_name,
    city,
    rating,
    COUNT(trip_id) AS total_trips,
    SUM(completion_flag) AS completed_trips,
    SUM(cancellation_flag) AS cancelled_trips,
    ROUND(AVG(delay_minutes), 2) AS avg_delay_minutes,
    ROUND(SUM(fare_amount), 2) AS total_revenue,
    ROUND(
        SUM(completion_flag) * 100.0 / COUNT(trip_id),
        2
    ) AS completion_rate,
    ROUND(
        SUM(cancellation_flag) * 100.0 / COUNT(trip_id),
        2
    ) AS cancellation_rate
FROM silver_ride_data
GROUP BY
    driver_id,
    driver_name,
    city,
    rating
ORDER BY total_revenue DESC;

driver_id,driver_name,city,rating,total_trips,completed_trips,cancelled_trips,avg_delay_minutes,total_revenue,completion_rate,cancellation_rate
114,Neha_114,Bangalore,3.7,4,3,1,11.25,720.44,75.00,25.00
107,Rahul_107,Mumbai,3.9,3,2,1,11.33,530.96,66.67,33.33
65,Amit_65,Bangalore,3.6,3,3,0,9.0,493.75,100.00,0.00
102,Vikas_102,Delhi,4.8,2,2,0,7.0,390.52,100.00,0.00
132,Priya_132,Hyderabad,4.0,2,2,0,19.0,387.74,100.00,0.00
76,Priya_76,Delhi,4.2,3,3,0,9.67,369.47,100.00,0.00
20,Vikas_20,Mumbai,4.1,3,2,1,10.0,336.66,66.67,33.33
63,Amit_63,Delhi,4.6,1,1,0,13.0,334.21,100.00,0.00
112,Amit_112,Delhi,4.5,2,1,1,7.0,326.4,50.00,50.00
103,Vikas_103,Bangalore,3.7,3,2,1,8.33,256.07,66.67,33.33


In [0]:
%sql
SELECT
    driver_id,
    driver_name,
    city,
    COUNT(trip_id) AS total_trips,
    SUM(cancellation_flag) AS cancelled_trips,
    ROUND(
        SUM(cancellation_flag) * 100.0 / COUNT(trip_id),
        2
    ) AS cancellation_rate
FROM silver_ride_data
GROUP BY
    driver_id,
    driver_name,
    city
ORDER BY
    cancellation_rate DESC,
    cancelled_trips DESC;

driver_id,driver_name,city,total_trips,cancelled_trips,cancellation_rate
91,Rahul_91,Pune,3,3,100.00
61,Rahul_61,Bangalore,3,3,100.00
126,Vikas_126,Hyderabad,2,2,100.00
57,Priya_57,Pune,2,2,100.00
88,Rohit_88,Pune,2,2,100.00
124,Priya_124,Mumbai,2,2,100.00
81,Amit_81,Mumbai,2,2,100.00
98,Neha_98,Bangalore,2,2,100.00
139,Vikas_139,Hyderabad,2,2,100.00
108,Rahul_108,Pune,2,2,100.00


In [0]:
%sql
SELECT
    city,
    COUNT(trip_id) AS total_trips,
    ROUND(SUM(fare_amount), 2) AS total_revenue,
    ROUND(AVG(fare_amount), 2) AS average_fare,
    ROUND(SUM(distance_km), 2) AS total_distance_km,
    ROUND(
        SUM(fare_amount) / COUNT(trip_id),
        2
    ) AS revenue_per_trip
FROM silver_ride_data
GROUP BY city
ORDER BY total_revenue DESC;

city,total_trips,total_revenue,average_fare,total_distance_km,revenue_per_trip
Delhi,30,2851.14,95.04,432.64,95.04
Bangalore,37,2535.43,68.53,526.65,68.53
Mumbai,32,2286.38,71.45,407.46,71.45
Pune,34,1493.44,43.92,480.54,43.92
Hyderabad,17,924.69,54.39,214.18,54.39


In [0]:
%sql
SELECT
    driver_id,
    driver_name,
    city,
    COUNT(trip_id) AS total_trips,
    ROUND(AVG(delay_minutes), 2) AS avg_delay_minutes,
    MAX(delay_minutes) AS max_delay_minutes,
    SUM(
        CASE
            WHEN delay_minutes > 0 THEN 1
            ELSE 0
        END
    ) AS delayed_trips,
    ROUND(
        SUM(
            CASE
                WHEN delay_minutes > 0 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(trip_id),
        2
    ) AS delay_rate
FROM silver_ride_data
GROUP BY
    driver_id,
    driver_name,
    city
ORDER BY avg_delay_minutes DESC;

driver_id,driver_name,city,total_trips,avg_delay_minutes,max_delay_minutes,delayed_trips,delay_rate
99,Rahul_99,Delhi,1,20.0,20.0,1,100.00
19,Rahul_19,Hyderabad,1,20.0,20.0,1,100.00
143,Anjali_143,Mumbai,1,19.0,19.0,1,100.00
137,Karan_137,Delhi,1,19.0,19.0,1,100.00
132,Priya_132,Hyderabad,2,19.0,20.0,2,100.00
85,Karan_85,Mumbai,1,19.0,19.0,1,100.00
111,Karan_111,Pune,1,18.0,18.0,1,100.00
87,Vikas_87,Delhi,1,18.0,18.0,1,100.00
142,Sneha_142,Delhi,1,16.0,16.0,1,100.00
45,Anjali_45,Delhi,1,16.0,16.0,1,100.00


In [0]:
%sql
WITH driver_metrics AS (
    SELECT
        driver_id,
        driver_name,
        city,
        rating,
        COUNT(trip_id) AS total_trips,
        SUM(completion_flag) AS completed_trips,
        SUM(cancellation_flag) AS cancelled_trips,
        ROUND(AVG(delay_minutes), 2) AS avg_delay_minutes,
        ROUND(SUM(fare_amount), 2) AS total_revenue,
        ROUND(
            SUM(completion_flag) * 100.0 / COUNT(trip_id),
            2
        ) AS completion_rate
    FROM silver_ride_data
    GROUP BY
        driver_id,
        driver_name,
        city,
        rating
)

SELECT
    *,
    DENSE_RANK() OVER (
        ORDER BY
            completion_rate DESC,
            total_revenue DESC,
            avg_delay_minutes ASC,
            rating DESC
    ) AS overall_driver_rank,

    DENSE_RANK() OVER (
        PARTITION BY city
        ORDER BY
            completion_rate DESC,
            total_revenue DESC,
            avg_delay_minutes ASC,
            rating DESC
    ) AS city_driver_rank

FROM driver_metrics
ORDER BY overall_driver_rank;

driver_id,driver_name,city,rating,total_trips,completed_trips,cancelled_trips,avg_delay_minutes,total_revenue,completion_rate,overall_driver_rank,city_driver_rank
65,Amit_65,Bangalore,3.6,3,3,0,9.0,493.75,100.00,1,1
102,Vikas_102,Delhi,4.8,2,2,0,7.0,390.52,100.00,2,1
132,Priya_132,Hyderabad,4.0,2,2,0,19.0,387.74,100.00,3,1
76,Priya_76,Delhi,4.2,3,3,0,9.67,369.47,100.00,4,2
63,Amit_63,Delhi,4.6,1,1,0,13.0,334.21,100.00,5,3
87,Vikas_87,Delhi,3.6,1,1,0,18.0,236.14,100.00,6,4
99,Rahul_99,Delhi,4.2,1,1,0,20.0,232.49,100.00,7,5
38,Sneha_38,Pune,5.0,2,2,0,8.5,223.44,100.00,8,1
125,Neha_125,Hyderabad,3.5,2,2,0,5.0,214.39,100.00,9,2
54,Anjali_54,Mumbai,3.7,1,1,0,2.0,212.08,100.00,10,1


## Spark SQL Analysis

The cleaned Silver dataset was registered as a temporary SQL view named `silver_ride_data`.

Spark SQL was used to perform:

- Driver performance analysis
- Cancellation rate analysis
- Revenue analysis
- Delay analysis
- Overall driver ranking
- City-wise driver ranking using window functions

This demonstrates the use of both PySpark DataFrame APIs and Spark SQL within the same data engineering pipeline.